In [ ]:
# Network segments to evaluate. Edit these before running.
# Enter any address in the target /24 (e.g. "10.1.5.0" or "10.1.5.0/24").
oco = "X.X.X.0"   # OCO: the subnet YOUR group attacked
dco = "X.X.X.0"   # DCO: YOUR group's subnet


In [ ]:
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
import traceback
import sys
import datetime
import pandas as pd
import warnings
warnings.filterwarnings("ignore")   # hide Python warnings; errors still show
from IPython.display import display

from df_binning import *
from generate_individual_labels_dataframe import *

#############
# Run variables
#############

benign_label = "none"

conn_server_loc = '/home/kali/datasets/CNT4416_full_csv'

log_location = "/home/kali/datasets/randomforest/logs/randomForest/" + str(datetime.date.today()) + "_log.txt"

########################
# functions
########################

def segment_prefix(addr):
    """First three octets plus a trailing dot, e.g. '10.1.5.0/24' -> '10.1.5.'"""
    octets = addr.strip().split("/")[0].split(".")
    if len(octets) < 3 or not all(o.isdigit() and 0 <= int(o) <= 255 for o in octets[:3]):
        raise ValueError("'" + addr + "' is not a valid IPv4 network segment")
    return ".".join(octets[:3]) + "."

def segment_confusion_matrix(preds, ip_col, prefix):
    """2x2 matrix (rows = actual, cols = predicted; order 0.0, 1.0) counting only the
    rows whose ip_col falls in the /24. Always 2x2, even if one class is absent."""
    subset = preds.filter(col(ip_col).cast("string").startswith(prefix))
    counts = {(r["label_bin"], r["prediction"]): r["count"]
              for r in subset.groupBy("label_bin", "prediction").count().collect()}
    vals = [0.0, 1.0]
    return np.array([[counts.get((a, p), 0) for p in vals] for a in vals], dtype=int)

oco_prefix = segment_prefix(oco)
dco_prefix = segment_prefix(dco)
print("OCO segment (dest_ip): " + oco_prefix + "x")
print("DCO segment (dest_ip): " + dco_prefix + "x")

########################
# code start
########################

spark = ( 
    SparkSession
    .builder
    .master("local[*]")
    .appName("TM_RandoForest")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
    )

# Only show Spark errors (hides WARN/INFO messages such as maxToStringFields)
spark.sparkContext.setLogLevel("ERROR")

printToLog("Begin run\n-----------\n----------\n\n", log_location)
printToLog("Dataset location - " + conn_server_loc, log_location)

source_df = spark.read.option("header", True).options(inferSchema=True).csv(conn_server_loc)
 
try:
    source_df = rename_columns(source_df)
except Exception as e:
    print("Error:  " + str(e), file=sys.stderr)
    print(traceback.format_exc())   

# Binary label: any technique label other than "none" = attack (0.0), "none" = benign (1.0)
unbinned_df = source_df.withColumn("label_bin", when((col("label_multi") != benign_label), 0.0).otherwise(1.0))
printToLog(str(unbinned_df.count()) + " rows", log_location)

#Carry out the binning on the df
numeric_percent_trim = 0.02
nominal_percent_agg = 0.98
replace_bool = False

attrList_ip_addr = ['dest_ip','src_ip']
attrList_port = ['dest_port', 'src_port']
attrList_bool = ['local_orig', 'local_resp']
attrList_nominal = ['protocol', 'conn_state', 'history','service']
attrList_numeric = ['duration','orig_bytes','orig_pkts','orig_ip_bytes','resp_bytes','resp_pkts','resp_ip_bytes','missed_bytes']

printToLog("Binning start", log_location)
conn_df = genFullBinnedDF(unbinned_df, attrList_ip_addr, attrList_port, attrList_bool, attrList_nominal, nominal_percent_agg, attrList_numeric, numeric_percent_trim, replace_bool).persist()
printToLog("Binning finished", log_location)

# balanced class weights rarer classes get a higher weight so the
# model is penalized more for missing them. Same formula sklearn uses for
# class_weight='balanced': total / (num_classes * class_count).
class_counts = conn_df.groupBy("label_bin").count()
total_count = conn_df.count()
num_classes = class_counts.count()
class_counts = class_counts.withColumn("class_weight", total_count / (num_classes * col("count")))
conn_df = conn_df.join(class_counts.select("label_bin", "class_weight"), on="label_bin", how="left").persist()

########################
# Build Feature cols
########################     

feature_cols = []
for clm, types in conn_df.dtypes:
    if "_bin" in clm:
        if not "label" in clm:
            feature_cols.append(clm)

# build feature vector for Classifier
assembler = VectorAssembler(inputCols = feature_cols, outputCol = "features")
conn_df = assembler.transform(conn_df)

printToLog("feature_cols - " + str(feature_cols), log_location)
print(str(len(feature_cols)) + " features: " + str(feature_cols))

########################
# Model: trained on every connection EXCEPT those going to the OCO and DCO subnets,
# so the model is scored on traffic it has never seen
########################   

in_segments = coalesce(col("dest_ip").cast("string").startswith(oco_prefix) |
                       col("dest_ip").cast("string").startswith(dco_prefix), lit(False))
train_df = conn_df.filter(~in_segments)
score_df = conn_df.filter(in_segments)

rf = RandomForestClassifier(featuresCol = "features", labelCol = "label_bin", weightCol = "class_weight")
rfModel = rf.fit(train_df)
printToLog("randomForest model fit", log_location)

#############
# Predictions on the OCO and DCO connections only
#############   

predictions = rfModel.transform(score_df)

for ip_c in ("dest_ip",):
    if ip_c not in predictions.columns:
        raise RuntimeError("'" + ip_c + "' is not in the binned dataframe; segment filtering needs it")

################################
# Output: confusion matrices counting only connections destined to the OCO and DCO subnets
################################

segments = [("OCO", "dest_ip", oco_prefix), ("DCO", "dest_ip", dco_prefix)]

# Same table format as TM_ML.randForestMaster. Positive = attack (label_bin 0.0).
for seg_name, ip_col, prefix in segments:
    cm = segment_confusion_matrix(predictions, ip_col, prefix)
    tp, fn = int(cm[0, 0]), int(cm[0, 1])   # actual attack: predicted attack / predicted benign
    fp, tn = int(cm[1, 0]), int(cm[1, 1])   # actual benign: predicted attack / predicted benign

    two_by_two = pd.DataFrame(
        [[f"TP: {tp}", f"FN: {fn}"],
         [f"FP: {fp}", f"TN: {tn}"]],
        index=["Actual: Attack", "Actual: Benign"],
        columns=["Predicted: Attack", "Predicted: Benign"]
    )

    print(f"\n{seg_name}:")
    print(f"  {ip_col} {prefix}x  |  {tp + fn + fp + tn:,} connections")
    display(two_by_two)

    # Same formulas as TM_ML.randForestMaster, with attack as the positive class
    total = tp + fn + fp + tn
    accuracy = (tp + tn) / total if total > 0 else float("nan")
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else float("nan"))
    print(f"  Accuracy: {accuracy:.1%}   "
          f"Recall: {recall:.1%}   "
          f"F1: {f1:.1%}")
    if tp + fn + fp + tn == 0:
        print(f"  No connections matched {prefix}x - check the {seg_name} address.")

conn_df.unpersist()
printToLog("End run\n-----------\n----------\n\n", log_location)
